<a href="https://colab.research.google.com/github/honestfarmer-cod/greends-pml/blob/main/Assig_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/YOUR_REPO_NAME/blob/main/YOUR_NOTEBOOK_PATH.ipynb)

In [18]:
import gradio as gr
from PIL import Image
import torch
import torch.nn as nn

# Redefine the Net0 model class (same as b223fc94) to load the weights
class Net0(nn.Module):
    def __init__(self):
        super(Net0, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the model and load the saved weights
deploy_model = Net0()
deploy_model.load_state_dict(torch.load('mnist_net01_finetuned.pth', map_location=torch.device('cpu')))

# Define device within this cell for deployment context
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

deploy_model.eval() # Set to evaluation mode
deploy_model.to(device) # Move to the correct device (CPU/GPU)

print("Model loaded successfully for deployment.")

Model loaded successfully for deployment.


In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

Next, we'll define transformations for the MNIST dataset, including resizing, converting to tensor, and normalization. We'll then load the training and test datasets and create data loaders.

In [7]:
# Define transformations for the MNIST dataset for a 28x28 1-channel input custom network

transform = transforms.Compose([
    transforms.ToTensor(), # Converts PIL Image to PyTorch Tensor (already 1 channel)
    transforms.Normalize((0.1307,), (0.3081,)), # MNIST specific normalization for 1 channel
    # transforms.Flatten() # Removed: 'Flatten' is not available in this torchvision.transforms version
])

# Load MNIST training and test datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# Create data loaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Training dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Number of batches in training loader: {len(train_loader)}")
print(f"Number of batches in test loader: {len(test_loader)}")

100%|██████████| 9.91M/9.91M [00:00<00:00, 20.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 528kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.63MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 16.2MB/s]

Training dataset size: 60000
Test dataset size: 10000
Number of batches in training loader: 938
Number of batches in test loader: 157


Now, let's set up the pre-trained convolutional neural network (CNN) model. We'll use ResNet-18 as an example, load its pre-trained weights, freeze all layers except the final classification layer, and then replace the final layer to match the number of classes in MNIST (10 classes).

In [8]:
# Define a lightweight custom Convolutional Neural Network (CNN) for MNIST
class Net0(nn.Module):
    def __init__(self):
        super(Net0, self).__init__()
        # Convolutional Layer 1
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1) # Input 1 channel, output 32 channels, 3x3 kernel
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # Reduces 28x28 to 14x14

        # Convolutional Layer 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # Input 32 channels, output 64 channels
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # Reduces 14x14 to 7x7

        # Fully Connected Layers
        # Input features: 64 channels * 7 * 7 (from previous pooling layer)
        self.fc1 = nn.Linear(64 * 7 * 7, 128) # First fully connected layer
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)    # Output layer for 10 MNIST classes

    def forward(self, x):
        # Apply Conv -> ReLU -> Pool
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))

        # Flatten the feature maps for the fully connected layer
        x = x.view(-1, 64 * 7 * 7)

        # Apply Fully Connected Layers
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the custom Net0 model
model = Net0()

# Move the model to the appropriate device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(f"Custom CNN Net0 model loaded and moved to {device}.")
print(model)

Custom CNN Net0 model loaded and moved to cuda.
Net0(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu1): ReLU()
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu2): ReLU()
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (relu3): ReLU()
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


Next, we define the loss function and the optimizer for training. We'll use Cross-Entropy Loss, which is common for multi-class classification, and the Adam optimizer.

In [9]:
# Define loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001) # Optimize all parameters of the custom Net0

print("Loss function (CrossEntropyLoss) and Adam optimizer initialized.")

Loss function (CrossEntropyLoss) and Adam optimizer initialized.


Finally, we will define the training and evaluation loops. The training loop will iterate through the training data, perform a forward pass, calculate the loss, perform a backward pass, and update the model's weights. The evaluation loop will assess the model's performance on the test data.

In [10]:
# Training loop
num_epochs = 2 # Changed to 2 epochs as requested

for epoch in range(num_epochs):
    model.train() # Set model to training mode
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device);

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train

    # Evaluation loop
    model.eval() # Set model to evaluation mode
    correct_test = 0
    total_test = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total_test += labels.size(0)
            correct_test += (predicted == labels).sum().item()

    test_acc = correct_test / total_test

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.4f}, "
          f"Test Acc: {test_acc:.4f}")

print("Fine-tuning complete!")

# Save the fine-tuned model weights
torch.save(model.state_dict(), 'mnist_net01_finetuned.pth')
print("Model weights saved to 'mnist_net01_finetuned.pth'")

Epoch 1/2, Train Loss: 0.1263, Train Acc: 0.9613, Test Acc: 0.9872
Epoch 2/2, Train Loss: 0.0419, Train Acc: 0.9872, Test Acc: 0.9884
Fine-tuning complete!
Model weights saved to 'mnist_net01_finetuned.pth'


## Deploying the Model with Gradio

Now that the model is trained and its weights are saved, we can deploy it using Gradio to create an interactive web interface. This will allow you to upload an image of a handwritten digit and get predictions from your trained `Net0` model.

In [11]:
# Install Gradio
!pip install -q gradio

In [17]:
import gradio as gr
from PIL import Image
import torch
import torch.nn as nn

# Redefine the Net0 model class (same as b223fc94) to load the weights
class Net0(nn.Module):
    def __init__(self):
        super(Net0, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = x.view(-1, 64 * 7 * 7)
        x = self.relu3(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the model and load the saved weights
deploy_model = Net0()
deploy_model.load_state_dict(torch.load('mnist_net01_finetuned.pth', map_location=torch.device('cpu')))

# Define device within this cell for deployment context
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

deploy_model.eval() # Set to evaluation mode
deploy_model.to(device) # Move to the correct device (CPU/GPU)

print("Model loaded successfully for deployment.")

Model loaded successfully for deployment.


In [25]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import gradio as gr # Import gradio for gr.Error

# Define the prediction function for Gradio
def predict_digit(sketchpad_input, image_upload_input):
    image = None

    # Handle sketchpad input
    if sketchpad_input is not None:
        if isinstance(sketchpad_input, Image.Image):
            image = sketchpad_input
        elif isinstance(sketchpad_input, dict) and 'composite' in sketchpad_input:
            image = sketchpad_input['composite']

    # Handle image upload input if sketchpad did not provide a valid image
    if image is None and image_upload_input is not None:
        if isinstance(image_upload_input, Image.Image):
            image = image_upload_input
        elif isinstance(image_upload_input, dict) and 'composite' in image_upload_input:
            image = image_upload_input['composite']

    if image is None:
        raise gr.Error("Please provide an input image, either by drawing or uploading.")

    # Ensure the image is grayscale (1 channel) and resized before applying transformations
    # Now 'image' is guaranteed to be a PIL.Image.Image object here
    image = image.convert('L') # Force conversion to 1-channel grayscale

    # Define transformations for preprocessing
    preprocess_transform = transforms.Compose([
        transforms.Resize((28, 28)), # Resize to 28x28 pixels
        transforms.ToTensor(), # Converts PIL Image (now 'L' mode) to PyTorch Tensor [1, H, W]
    ])

    image_tensor = preprocess_transform(image)

    # Explicitly check and handle 3-channel case, just in case ToTensor still produces it
    if image_tensor.shape[0] == 3:
        # If it's still 3 channels, average them to get 1 channel
        image_tensor = image_tensor.mean(dim=0, keepdim=True) # Converts [3, H, W] to [1, H, W]

    # Apply normalization
    normalize_transform = transforms.Normalize((0.1307,), (0.3081,))
    image_tensor = normalize_transform(image_tensor)

    # Add batch dimension and move to device
    image_tensor = image_tensor.unsqueeze(0).to(device)

    # Debug print statement for input tensor shape
    print(f"DEBUG: Input image_tensor shape before model: {image_tensor.shape}")

    # Make a prediction
    with torch.no_grad():
        outputs = deploy_model(image_tensor)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

    # Get the predicted class and its probability
    predicted_prob, predicted_idx = torch.max(probabilities, 0)

    # Map to digit labels
    labels = [str(i) for i in range(10)]

    # Return the predicted class and all probabilities as a dictionary for Gradio
    return {labels[i]: float(probabilities[i]) for i in range(10)}

print("Prediction function defined (with forced grayscale, resize, and explicit channel check, now supporting multiple inputs).")

Prediction function defined (with forced grayscale, resize, and explicit channel check, now supporting multiple inputs).


In [26]:
# Create and launch the Gradio interface
interface = gr.Interface(
    fn=predict_digit,
    inputs=[
        gr.Sketchpad(type="pil", label="Draw a handwritten digit"),
        gr.Image(type="pil", label="Or Upload an image of a handwritten digit (28x28 grayscale works best)")
    ],
    outputs=gr.Label(num_top_classes=3, label="Prediction"), # Show top 3 predicted classes
    title="MNIST Digit Classifier (Net0 CNN)",
    description="Draw a handwritten digit on the canvas OR upload an image, and let the Net0 CNN predict what it is!",
    examples=[
        # You can add example image paths here if you have them
        # For instance, a path to an image of '0', '1', etc.
        # 'path/to/digit_0.png', 'path/to/digit_1.png'
    ]
)

print("Launching Gradio interface...")
interface.launch(debug=True, share=True)

Launching Gradio interface...
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7bf73321721f548628.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


DEBUG: Input image_tensor shape before model: torch.Size([1, 1, 28, 28])
Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://7bf73321721f548628.gradio.live
